In [2]:
from dataclasses import dataclass

import h5py
import pandas
from pandas import DataFrame

LSL_STREAMS = {
    "MendiLSL": [
        "acc_x", "acc_y", "acc_z",
        "ang_x", "ang_y", "ang_z",
        "temp",
        "ir_l", "red_l", "amb_l",
        "ir_r", "red_r", "amb_r",
        "ir_p", "red_p", "amb_p",
        "battery_voltage", "timestamp"
    ],
    "OxySoft": [
        "rx1_l1", "rx1_l2", "rx1_l3", "rx1_l4",
        "rx1_l5", "rx1_l6", "rx1_l7", "rx1_l8",
        "rx1_l9", "rx1_l10", "rx1_l11", "rx1_l12",
        "rx1_l13", "rx1_l14", "rx1_l15", "rx1_l16",
        "rx2_l1", "rx2_l2", "rx2_l3", "rx2_l4",
        "rx2_l5", "rx2_l6", "rx2_l7", "rx2_l8",
        "rx2_l9", "rx2_l10", "rx2_l11", "rx2_l12",
        "rx2_l13", "rx2_l14", "rx2_l15", "rx2_l16",
        "battery_voltage", "timestamp"
    ]
}

def h5_stream_to_df(filepath, columns) -> pandas.DataFrame:
    with h5py.File(filepath, "r") as h:
        return pandas.DataFrame(h["stream"], columns=columns)

def h5_events_to_df(filepath) -> pandas.DataFrame:
    with h5py.File(filepath, "r") as h:
        return pandas.DataFrame.from_records(h["events"][:])

mendi_stream_df = h5_stream_to_df("data/participant4-mendi.h5", LSL_STREAMS["MendiLSL"])
artinis_stream_df = h5_stream_to_df("data/participant4-artinis.h5", LSL_STREAMS["OxySoft"])

mendi_events_df = h5_events_to_df("data/participant4-mendi.h5")
artinis_events_df = h5_events_to_df("data/participant4-artinis.h5")


# Data Cleaning

In [33]:
def clean_events(events_df: DataFrame) -> DataFrame:
    """

    :param events_df: the events DataFrame to clean
    :return:the cleaned up events DataFrame
    """

    keep = []
    prev_switch = ""

    for index, row in list(events_df.iterrows()):

        if row.event != b"switch":

            #
            if prev_switch == b"rest.html":
                keep.append(False)
            else:
                keep.append(True)

        else:

            #
            if row.value == prev_switch:
                keep.append(False)
            else:
                keep.append(True)

            #
            prev_switch = row.value

    # Add missing front rest label due to initial refreshes

    #
    return events_df[keep].reset_index(drop=True)

def clean_stream(stream: pandas.DataFrame) -> pandas.DataFrame:
    # TODO: apply device specific stream cleaning
    pass


clean_mendi_events_df = clean_events(mendi_events_df)
clean_artinis_events_df = clean_events(artinis_events_df)


# Modified Beer-Lambert Law

In [35]:
from itertools import pairwise

def rest_baseline(rest_df):
    return 0.0

@dataclass
class Block:
    i_baseline: float
    stream: DataFrame

def split_stream(stream_df, events_df):
    switch_events = list(events_df.loc[
        events_df["event"].eq(b"switch")
    ].itertuples())

    rest_events = list(events_df.loc[
        events_df["event"].eq(b"switch")
        & events_df["value"].eq(b"rest.html")
    ].itertuples())

    # T
    blocks = []

    for current, baseline_end in pairwise(rest_events):
        next_switch = next(
            (
                event for event in switch_events
                if event.timestamp > current.timestamp
            ),
            switch_events[-1],
        )

        i_baseline = rest_baseline(stream_df.loc[
            stream_df["timestamp"].ge(current.timestamp)
            & stream_df["timestamp"].lt(baseline_end.timestamp)
        ])

        block_stream = stream_df.loc[
            stream_df["timestamp"].ge(current.timestamp)
            & stream_df["timestamp"].lt(next_switch.timestamp)
        ]

        blocks.append(Block(i_baseline, block_stream))

    return blocks



split_stream_df = split_stream(artinis_stream_df, clean_artinis_events_df)

[Block(i_baseline=0.0, stream=      rx1_l1   rx1_l2   rx1_l3   rx1_l4   rx1_l5   rx1_l6   rx1_l7   rx1_l8  \
124  0.91450  0.91675  0.93625  0.87850  1.04250  1.07100  0.30100  0.22125   
125  0.91700  0.91850  0.94000  0.88100  1.04525  1.07275  0.30375  0.22325   
126  0.91650  0.91800  0.94300  0.88300  1.04675  1.07325  0.30475  0.22400   
127  0.91400  0.91625  0.94275  0.88250  1.04750  1.07400  0.30450  0.22375   
128  0.91300  0.91550  0.94125  0.88150  1.04775  1.07450  0.30375  0.22325   
..       ...      ...      ...      ...      ...      ...      ...      ...   
719  0.91025  0.90425  0.93125  0.87475  1.04750  1.07300  0.30325  0.22075   
720  0.91175  0.90500  0.93325  0.87625  1.04850  1.07325  0.30450  0.22150   
721  0.91175  0.90500  0.93250  0.87550  1.04825  1.07325  0.30425  0.22125   
722  0.91175  0.90450  0.93200  0.87475  1.04825  1.07300  0.30400  0.22125   
723  0.91100  0.90425  0.93100  0.87425  1.04750  1.07275  0.30325  0.22075   

      rx1_l9  rx1_l10

# MBLL

In [17]:
# Break into blocks


@dataclass
class Block:
    a: float
    epochs: list[DataFrame]

def blocks(stream_df, events_df):
    """

    :param events_df:
    :param stream_df:
    :return:
    """

    target_trial_df = events_df.loc[
        (events_df["value"] == b"TF") |
        (events_df["value"] == b"TS")
    ]

    epochs = []

    for index, row in target_trial_df.iterrows():
        epochs.append(stream_df.loc[
            (stream_df["timestamp"] >= row.timestamp - 30) &
            (stream_df["timestamp"] < row.timestamp + 10)
        ])

    return epochs



SyntaxError: invalid syntax (2999946788.py, line 34)